In [ ]:
# DDGR20，嵌入近似hint，l-noisy < <v,s> < l+noisy
cd framework

In [ ]:
load("../framework/instance_gen.sage")
import numpy as np
import random

In [ ]:
n = 512
m = n
q = 3329
D_s = build_centered_binomial_law(3)
D_e = D_s
A, b, s, dbdd = initialize_from_LWE_instance(DBDD_optimized, n, q, m, D_e, D_s)
# _ = dbdd.integrate_q_vectors(q, report_every=20)
beta, delta = dbdd.estimate_attack()

In [ ]:
def generate_se_eta_bias_approx_hint(m, n, bias, k):
    V = []
    L = []

    for i in range(k):
        D_e = {-3: 1/64, -2: 6/64, -1: 15/64, 0: 20/64, 1: 15/64, 2: 6/64, 3: 1/64}
        values, probabilities = zip(*D_e.items())
        v = np.array(np.random.choice(values, size=m+n, p=probabilities))
        noisy = random.randint(-bias, bias)
        l = dbdd.leak(v)+noisy
        V.append(v)
        L.append(l)
    print("L",L)
    return V,L

In [ ]:
nph_Kyber512 = [0, 40, 80, 120, 160, 200, 240, 280, 320, 360, 400, 440, 480, 520, 560, 600, 640, 680, 720, 760, 800, 840, 880, 920, 960, 1000, 1500, 2000, 2500, 3000, 3500, 4000]
dis_Kyber512_our = [39.56, 40.03, 40.4, 40.4, 40.31, 40.09, 39.85, 39.7, 39.23, 39.2, 38.86, 38.64, 38.33, 37.92, 37.5, 37.2, 37.1, 36.7, 36.54, 36.23, 35.97, 35.76, 35.48, 35.16, 34.96, 34.77, 31.73, 28.8, 26.84, 24.98, 23.22, 21.74]
num_hint = 1001
bias = int(q/32)
V, L = generate_se_eta_bias_approx_hint(m, n, bias, num_hint)
BETA_ori = []
BETA_com = []
index = 0
for j in range(num_hint):
    if j == nph_Kyber512[index]:
        # _ = dbdd.integrate_q_vectors(q, report_every=20)
        beta_ori, delta = dbdd.estimate_attack()
        print("beta_ori: ", beta_ori)
        BETA_ori.append(beta_ori)
        st = dis_Kyber512_our[index]/dis_Kyber512_our[0]
        beta_com, delta = dbdd.estimate_attack_SMY(st)
        print("beta_com: ", beta_com)
        BETA_com.append(beta_com)
        index += 1
    print("the ",j+1,"-th secret error sca approx hint")
    _ = dbdd.integrate_approx_hint(vec(V[j]), L[j], bias, aposteriori=False)
print("BETA_ori",BETA_ori)
print("BETA_com",BETA_com)